# Flag Algebras in Practice: Short tutorial

**Zászló** is a Python library implementing the *semidefinite method* for flag algebras.
Starting from a problem specification (which graphs are forbidden, what density to bound?), it
enumerates combinatorial structures, builds a semidefinite program (SDP), solves it, and
verifies the certificate in exact rational arithmetic.

In this tutorial, we'll use Mantel's theorem (surprise surprise) as a vehicle to show how to set up a Turán-type problem and interpret the results.

## Getting set up

Running the cell below will let us use the tools we need. It contains the barebones essentials for this library: click it, then press shift+enter to execute the cell. You'll need to do the same for all later code cells (they have that grey background, you can't miss 'em).

In [1]:
from fractions import Fraction

from zaszlo import (
    FlagProblem,
    Hypergraph,
    build_flag_algebra_data,
    certify,
    identify_sharps,
    solve_sdp,
)

Okay, now, let's state our problem.

---
## 1. Mantel's Theorem

**Theorem (Mantel, 1907).** A triangle-free graph on $n$ vertices has at most $\lfloor n^2/4 \rfloor$
edges.  Equivalently, its edge density satisfies $p(K_2; G) \le \frac{1}{2}$, with equality for
the balanced complete bipartite graph $K_{\lfloor n/2\rfloor,\lceil n/2\rceil}$.

We will recover the bound $\frac{1}{2}$ via flag algebras working at order $n = 4$.

### Defining the Parameters

We'll first define the forbidden graph $K_3$: Some shorthand exists to make this graph, but this is the most general way to define a graph in Zászló.

In [12]:
K3 = Hypergraph(4, 2, [(1, 2), (1, 3), (2, 3),(1,4)])

Here, we constructed the graph by telling Zászló the number of vertices, uniformity of edges, and the edges themselves. For a little more information, run the cell below:

In [13]:
K3

Hypergraph(n=4, k=2, edges=[(1, 2), (1, 3), (1, 4), (2, 3)])

**Tip — self-describing objects.** Every Zászló object has a `.explain()` method returning a plain-text description (works in scripts, REPLs, and notebooks alike). In Jupyter, objects also render directly as annotated HTML cards — Jupyter picks this up automatically via `_repr_mimebundle_`. Try putting any object on the last line of a cell, or call `display(obj)`, to see the visual form. We'll be doing this throughout the tutorial!

Alright, now let's see how to encode our problem (maximize edges, forbid $K_3$) correctly:

In [14]:
# Problem specification
prob_mantel = FlagProblem(
    4,              # n: admissible graphs on 4 vertices
    2,              # type_order: use types of order 0 and 2 (both even, like n=4)
    2,              # k: ordinary graphs (2-uniform)
    forbidden=[K3], # forbid non-induced K3
    minimize=False, # find an upper bound (minimize lambda)
)

Some quick notes: 

-`n` and `type_order` are variables you have some control over. Making these values larger will be more computationally intensive. `n` should probably be a single digit if you want the solver to ever find an answer.

-You also need $0\leq$ `type_order` $\leq$ `n`, with `type_order` and `n` of the same parity.

-Last, the format of `forbidden` is a list, and you can feed (comma-separated) entries to forbid more graphs.

As above, run the following cell to display some info about our problem:

In [15]:
prob_mantel

FlagProblem(n=4, type_order=2, k=2, minimize=False)

### Building Flags

We now need to take these constraints and generate some flags! The next two cells should help with that. This object is quite rich, check out the longer tutorial to see more.

In [16]:
data_mantel = build_flag_algebra_data(prob_mantel)

In [17]:
print(data_mantel.explain())

FlagAlgebraData  (pre-SDP combinatorial data)

  Types:             3  (one Q matrix / SDP block each)
  Flags per type:    [2, 4, 4]  (10 total)
  Admissible graphs: 8  (objects whose densities are bounded)
  Density range:     [0.0000, 0.6667]

This is the input to the SDP solver. The solver searches for PSD
matrices Q_σ (one per type σ) satisfying, for every admissible H:

  bound − density(H)  =  Σ_σ ⟨Q_σ, P_σ(H)⟩  +  slack(H)

The PSD matrices Q_σ define a flag-algebra sum-of-squares expression.
Together with the verified coefficient inequalities over admissible
graphs, this yields the stated bound.

  Admissible graphs — the feasible graphs on n vertices; the SDP
    bounds their density.  Forbidden subgraphs have already been
    filtered out.
  Types — all-labeled graphs; each type σ indexes one SDP block Q_σ.
  Flags — graphs with a type embedded; pairs of flags over σ average
    to admissible densities, giving the constraint matrix P_σ(H).


### Solving the SDP

`solve_sdp` uses Clarabel to find the optimal $\lambda$ and PSD matrices $\{Q_\sigma\}$.
`extract_Q=True` stores the certificate matrices so we can verify them afterwards. Run the following cell to solve the SDP. It should be pretty fast here, since this problem is small.

In [18]:
result_mantel = solve_sdp(data_mantel, extract_Q=True)

print(result_mantel.explain())

FlagAlgebraResult

  Bound  : 0.50000000  (upper bound)
  Status : optimal

Proof claim: Every graph avoiding the 1 forbidden pattern(s) has edge density ≤ 0.500000.

Sharp graphs (active constraints):
  3 graph(s) with near-zero SDP slack.
  These graphs have tight certificate constraints. Note: zero slack
  does not by itself mean a graph attains the density bound.

    [0]  4v 2-uniform hypergraph, 0 edges  (density = 0.0000)
    [4]  4v 2-uniform hypergraph, 3 edges  (density = 0.5000)
    [7]  4v 2-uniform hypergraph, 4 edges  (density = 0.6667)

Proof certificate:
  The bound is certified by PSD matrices Q_σ, one per type σ.
  For every admissible H the following identity holds and is ≥ 0:

    bound − density(H)  =  Σ_σ ⟨Q_σ, P_σ(H)⟩  +  slack(H)

  where:
    P_σ(H) — pair density matrix of H over type σ
    Q_σ    — PSD certificate matrix (one per type, found by SDP)
    ⟨A, B⟩ — matrix inner product  Σ_{ij} A_{ij} B_{ij}

  The PSD matrices Q_σ define a flag-algebra sum-of-sq

### Sharp (extremal) graphs

An admissible graph $H$ is *sharp* when its SDP slack is zero: the bound is tight at $H$.
Sharps are the graphs that appear in the extremal construction — for Mantel, the extremal
graphon is the balanced bipartite graphon ($K_{n/2,n/2}$ in the limit).

In [9]:
sharps_mantel = identify_sharps(data_mantel, result_mantel)


sharps_mantel

SharpsResult  3 sharp graph(s)  (n=4, k=2)

## Getting an exact certificate

The SDP solver returns floating-point $Q$ matrices. `certify()` rounds these to exact rational arithmetic via Cholesky factorization — guaranteeing the result is PSD by construction — then verifies all residuals are non-negative. It returns a `Certificate` object with the exact certified bound, residuals, and a validity flag.

In [10]:
proof_mantel = certify(result_mantel)

proof_mantel

Certificate  bound=24834471153474084/49668769112077837  [valid]

## For programmatic / agent use: `diagnose()`, JSON export, and the corpus

If you're driving Zászló from a script or an AI agent, three additional APIs make the workflow machine-friendly:

- **`result.diagnose()`** → a `DiagnosticReport` dataclass of structured observations (validity, active constraints, sharp densities, whether the extremal graph attains the bound, aux-constraint activity). No heuristic hints — just the facts, so the agent can reason over them.
- **`result.to_json()` / `result.certificate.to_json()`** → exact-rational proof certificate as JSON, ready for storage, transmission, or downstream verification.
- **`zaszlo.corpus`** → 10 curated reference problems (Mantel, Turán K₄/K₅, pentagon C₅ density, K₄⁻-free 3-graphs, …) with citations and expected bounds. Pattern-match your problem against these before hand-crafting a `FlagProblem`.

Also see `SKILL.md` at the repo root — an agent-facing map of the whole library.

In [ ]:
from zaszlo import corpus

# Look up Mantel in the corpus (same problem we just solved by hand).
entry = corpus.get("mantel")
print(f"Name    : {entry.name}")
print(f"Citation: {entry.citation}")
print(f"Expected: {entry.expected_bound}")
print(f"Tags    : {entry.tags}")

# Structured diagnostic report from the certificate we just built.
diag = proof_mantel.diagnose()
print()
print(f"Bound       : {diag.bound_exact}   valid={diag.is_valid}")
print(f"Sharp graphs: {diag.sharp_indices}  densities={diag.sharp_densities}")
print(f"Extremal attains bound? {diag.max_sharp_density_matches_bound}")
print(f"Aux constraints: {diag.aux_summary}")

# JSON certificate (first 400 chars only).
print()
print("Certificate JSON (truncated):")
print(proof_mantel.to_json(indent=2)[:400] + "...")

Here ends our brief tutorial! This should get you started with any Turán-type problem you fancy. The longer tutorial will also walk you through using Zászló to find the maximum induced density of subgraphs beyond just the edge via the Pentagon problem, as well how to work with hypergraphs of higher uniformity. It also has more pictures!